<a href="https://colab.research.google.com/github/arnavm2k3/TorchCode-Solutions/blob/main/templates/11_sliding_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

### References
Sliding Window Attention - [Link](https://medium.com/@manojkumal/sliding-window-attention-565f963a1ffd)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.8 MB/s eta 0:00:00


In [2]:
import torch
import math

In [29]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
  B, seq, d_k = Q.size()
  mask = (1 - torch.triu(torch.ones(seq, seq), diagonal=window_size+1)) * torch.triu(torch.ones(seq, seq), diagonal=-window_size)
  scores = (torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)) * mask
  scores[scores == 0] = float('-inf')

  attention = torch.bmm(torch.softmax(scores, dim=-1), V)
  return attention

In [30]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [31]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.3ms)
  ✅ [2/5] window_size=0 — only sees itself (0.6ms)
  ✅ [3/5] Large window equals full attention (5.2ms)
  ✅ [4/5] Distant tokens don't affect output (1.7ms)
  ✅ [5/5] Gradient flow (0.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (9.5ms total)
  Progress saved. Run status() to see your dashboard.

